In [ ]:
# =========================================================
# 🚀 TXO IV + PCR 引擎 (修正輸出欄位版)
# 篩選條件：商品代號 == "TXO" 且 成交量 >= 30
# =========================================================
!pip install -q numba tqdm

import pandas as pd
import numpy as np
import zipfile, io, time, os
from pathlib import Path
from google.colab import drive
from tqdm import tqdm
from numba import njit, config
from math import log, sqrt, exp, erf

# ================= Numba & Drive 設定 =================
drive.mount('/content/drive', force_remount=True)
BASE_PATH = Path("/content/drive/MyDrive/金融資料探勘")
INDEX_FILE = BASE_PATH / "Index_411111211_2022.csv"
OPTION_ZIP = BASE_PATH / "Option_2022.zip"

# =========================================================
# 🚀 BLACK-SCHOLES & IV 計算核心
# =========================================================
@njit(fastmath=True)
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / 1.41421356237))

@njit(fastmath=True)
def bs_price(S, K, T, r, sigma, is_call=True):
    if sigma <= 0.0 or T <= 0.0:
        return max(S - K, 0.0) if is_call else max(K - S, 0.0)
    d1 = (log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    if is_call:
        return S * norm_cdf(d1) - K * exp(-r * T) * norm_cdf(d2)
    else:
        return K * exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)

@njit(fastmath=True)
def bisection_iv_vector(S, K, T, r, price, is_call=True):
    n = len(price)
    iv = np.empty(n, dtype=np.float64)
    for i in range(n):
        p = price[i]
        if p <= 0.01:
            iv[i] = np.nan
            continue
        low, high = 0.0001, 5.0
        for _ in range(50):
            mid = (low + high) / 2.0
            if bs_price(S, K[i], T, r, mid, is_call) < p: low = mid
            else: high = mid
        iv[i] = mid
    return iv

# =========================================================
# 🚀 ZIP 讀取器
# =========================================================
zip_cache = {}
def read_zip(target_csv):
    if target_csv in zip_cache: return zip_cache[target_csv]
    try:
        with zipfile.ZipFile(OPTION_ZIP) as z1:
            for inner in z1.namelist():
                if target_csv.replace(".csv", "") in inner:
                    with zipfile.ZipFile(io.BytesIO(z1.read(inner))) as z2:
                        with z2.open(z2.namelist()[0]) as f:
                            df = pd.read_csv(f, encoding="cp950", low_memory=False)
                            df.columns = [c.strip() for c in df.columns]
                            zip_cache[target_csv] = df
                            return df
    except: return None

# =========================================================
# 🚀 主計算邏輯
# =========================================================
index_df = pd.read_csv(INDEX_FILE)
index_df["Date"] = pd.to_datetime(index_df["Date"])

daily_results = []
start_t = time.time()

for _, row in tqdm(index_df.iterrows(), total=len(index_df), desc="🚀 計算中"):
    S0, r, contract = float(row["S0"]), float(row["Rf"]), str(row["Contract"]).strip()
    df = read_zip(row["File"])
    if df is None: continue

    # --- 1. 嚴格篩選: TXO + 指定月份 + 成交量 >= 30 ---
    sym_col = next((c for c in df.columns if "商品代號" in c), None)
    mon_col = next((c for c in df.columns if "到期月份" in c), None)
    vol_col = next((c for c in df.columns if "成交量" in c or "成交數量" in c), None)
    cp_col = next((c for c in df.columns if "買賣權" in c or "權別" in c), None)
    k_col = next((c for c in df.columns if "履約價" in c), None)
    p_col = next((c for c in df.columns if "成交價" in c), None)

    df[vol_col] = pd.to_numeric(df[vol_col], errors='coerce')
    mask = (df[sym_col].astype(str).str.strip() == "TXO") & \
           (df[mon_col].astype(str).str.strip() == contract) & \
           (df[vol_col] >= 30)
    df_filtered = df[mask].copy()

    if df_filtered.empty: continue

    # 數據轉型
    df_filtered[k_col] = pd.to_numeric(df_filtered[k_col], errors='coerce')
    df_filtered[p_col] = pd.to_numeric(df_filtered[p_col], errors='coerce')

    # --- 2. 計算 Call/Put IV ---
    T = 0.05 # 預設約 18 天到期

    # Call 統計
    df_c = df_filtered[df_filtered[cp_col].astype(str).str.contains("C|買")]
    iv_c = bisection_iv_vector(S0, df_c[k_col].values, T, r, df_c[p_col].values, True) if not df_c.empty else []

    # Put 統計
    df_p = df_filtered[df_filtered[cp_col].astype(str).str.contains("P|賣")]
    iv_p = bisection_iv_vector(S0, df_p[k_col].values, T, r, df_p[p_col].values, False) if not df_p.empty else []

    # --- 3. 收集指定欄位 ---
    vol_call = df_c[vol_col].sum() if not df_c.empty else 0
    vol_put = df_p[vol_col].sum() if not df_p.empty else 0

    daily_results.append({
        "Date": row["Date"].strftime('%Y-%m-%d'),
        "Total_Vol_Call": vol_call,
        "Num_Call": len(iv_c),
        "Mean_IV_Call": np.nanmean(iv_c) if len(iv_c) > 0 else np.nan,
        "Std_IV_Call": np.nanstd(iv_c, ddof=1) if len(iv_c) > 1 else np.nan,
        "Total_Vol_Put": vol_put,
        "Num_Put": len(iv_p),
        "Mean_IV_Put": np.nanmean(iv_p) if len(iv_p) > 0 else np.nan,
        "Std_IV_Put": np.nanstd(iv_p, ddof=1) if len(iv_p) > 1 else np.nan,
        "PCR_Volume": vol_put / vol_call if vol_call > 0 else np.nan
    })

# =========================================================
# 🚀 輸出結果
# =========================================================
output_df = pd.DataFrame(daily_results)
output_df.to_csv(BASE_PATH / "TXO_Daily_IV_PCR_Final.csv", index=False)

print(f"\n✅ 處理完成！耗時: {time.time()-start_t:.2f} 秒")
print("-" * 30)
print(output_df.head())

In [ ]:
import pandas as pd
from pathlib import Path

# 1. 設定路徑 (請確認您的雲端硬碟路徑)
BASE_PATH = Path("/content/drive/MyDrive/金融資料探勘")
FILE_PATH = BASE_PATH / "TXO_Daily_IV_PCR_Final.csv"
SAVE_PATH = BASE_PATH / "TXO_Daily_IV_PCR_Final_v2.csv" # 建議先存新檔確認

if FILE_PATH.exists():
    # 2. 讀取 CSV
    df = pd.read_csv(FILE_PATH)

    # 3. 指定需要統一位數的欄位
    float_cols = [
        "Mean_IV_Call", "Std_IV_Call",
        "Mean_IV_Put", "Std_IV_Put",
        "PCR_Volume"
    ]

    # 4. 執行四捨五入至小數點後 9 位
    df[float_cols] = df[float_cols].round(9)

    # 5. 儲存檔案
    # 使用 float_format='%.9f' 確保輸出的 CSV 文字檔裡也維持 9 位數
    df.to_csv(SAVE_PATH, index=False, float_format='%.9f')

    print(f"✅ 檔案位數修正完成！")
    print(f"📂 原始檔案: {FILE_PATH.name}")
    print(f"📂 修正檔案: {SAVE_PATH.name}")
    print("-" * 50)
    print("📊 修正後的數據預覽 (首行):")
    # 顯示第一筆數據確認位數
    print(df.head(1).to_string(index=False))
else:
    print("⚠️ 找不到檔案，請確認路徑是否正確。")